# LLM Laika
Antes de rodar: **Ambiente de execução → Alterar tipo de ambiente de execução → GPU (T4)**.

Tudo (livros, artigos, índice de busca, pré-treino, fine-tuning e memória) fica salvo no seu
Google Drive, na pasta `LLM_Laika`. Se a sessão cair, nada se perde.

| Quero... | Rode as células |
|---|---|
| Fazer tudo do zero | 1, 2, 3, 4, 5 |
| **Só refazer o fine-tuning** (usando o pré-treino que já está no Drive) | 1, 4, 5 |
| Atualizar livros/artigos e o índice de busca | 1, 2 |
| Só conversar | 1, 5 |
| Levar o modelo para o celular (app Laika) | 1, 6 |
| Medir o modelo (perguntas fixas, busca e perplexidade) e comparar com o anterior | 1, 7 |
| Dar um nome ao modelo no app (vai direto para os backups) | 1 com um NOME, 4 e/ou 6 |
| Treinar outro modelo sem apagar o atual (ex.: um pequeno) | 1 com um PROJETO, 3 (MODO "do_zero"), 4, 6 |
| **Ligar as células pelo app, sem deixar o Colab aberto** | 0 (uma vez só) |

Os campos ao lado das células (MODO, EPOCAS...) são os mesmos que o app mostra na tela **Treinar**.
Pelo app, a célula 1 sempre roda antes das outras; as células 0 e 5 só funcionam aqui no navegador.

In [ ]:
# 0) LIGAR O APP DA LAIKA AO COLAB [só no navegador] — rode uma vez só
# Depois disso, o app (menu ⋮ → Treinar) liga as células 1, 2, 3, 4 e 6 sem você abrir o Colab:
# o GitHub Actions do repositório roda o Colab CLI com a sua conta e desliga a máquina no fim.
# Antes de rodar:
#  a) crie um token em https://github.com/settings/personal-access-tokens/new (tipo "fine-grained"),
#     só para o repositório LLM_Laika, com as permissões Actions (leitura e escrita),
#     Secrets (leitura e escrita) e Contents (leitura). Guarde o token: ele também vai no app;
#  b) o arquivo .github/workflows/colab.yml precisa estar no branch principal do repositório.
# Ao rodar: abra o link que aparecer, entre com a sua conta Google, copie o código e cole aqui.
REPOSITORIO = "BrunoDevPro/LLM_Laika"  #@param {type:"string"}
import os, json, base64, getpass, urllib.request, urllib.error

escopos = ",".join(["openid", "https://www.googleapis.com/auth/cloud-platform",
                    "https://www.googleapis.com/auth/userinfo.email",
                    "https://www.googleapis.com/auth/colaboratory"])
!gcloud auth application-default login --no-launch-browser --scopes={escopos}
adc = os.path.join(os.environ.get("CLOUDSDK_CONFIG") or os.path.expanduser("~/.config/gcloud"),
                   "application_default_credentials.json")
if not os.path.exists(adc):
    raise RuntimeError("O login do Google não terminou. Rode a célula de novo.")

!pip install -q pynacl
from nacl import encoding, public
token = getpass.getpass("Token do GitHub (não aparece na tela): ").strip()

def github(metodo, caminho, dados=None):
    req = urllib.request.Request(
        f"https://api.github.com/repos/{REPOSITORIO}{caminho}", method=metodo,
        data=None if dados is None else json.dumps(dados).encode(),
        headers={"Authorization": "Bearer " + token, "Accept": "application/vnd.github+json",
                 "X-GitHub-Api-Version": "2022-11-28"})
    try:
        with urllib.request.urlopen(req) as r:
            corpo = r.read()
            return json.loads(corpo) if corpo else {}
    except urllib.error.HTTPError as e:
        dicas = {401: "o token está errado ou venceu",
                 403: "o token não tem a permissão necessária (veja o item a)",
                 404: "repositório não encontrado ou o token não tem acesso a ele"}
        raise RuntimeError(f"GitHub {e.code} em {caminho}: {dicas.get(e.code, e.reason)}") from None

chave = github("GET", "/actions/secrets/public-key")
selado = public.SealedBox(public.PublicKey(chave["key"].encode(), encoding.Base64Encoder()))
cifrado = base64.b64encode(selado.encrypt(open(adc, "rb").read())).decode()
github("PUT", "/actions/secrets/COLAB_ADC", {"encrypted_value": cifrado, "key_id": chave["key_id"]})
print("Pronto: a permissão do Colab foi guardada no GitHub (segredo COLAB_ADC).")
try:
    github("GET", "/actions/workflows/colab.yml")
    print("O workflow do Colab está no repositório. Agora é só usar a tela Colab do app.")
except RuntimeError:
    print("Atenção: o .github/workflows/colab.yml ainda não está no branch principal (faça o merge).")

In [ ]:
# 1) PREPARAR: GPU, Google Drive, código e atalhos para as pastas do Drive
# NOME: só o nome do modelo no app (ex.: "Laika 1.9 100M"). O zip da célula 6 leva o nome, e o app guarda o
# modelo direto nos backups com ele, sem trocar o modelo em uso. Não muda nenhuma pasta.
# PROJETO: só para um modelo NOVO, treinado do zero sem apagar o atual (ex.: "pequena 7M"). Com um nome, o
# pré-treino, o fine-tuning, a avaliação e os tokens ficam em pastas só dele (checkpoints_<projeto> e
# tokens_<projeto>); a pasta começa vazia, então rode a célula 3 antes da 4. Vazio = o modelo de sempre.
# Livros, artigos, índice do RAG e SFT continuam compartilhados. Sem NOME, o app usa o PROJETO como nome.
# PARTIR_DE: num PROJETO sem pré-treino, copia o pré-treino e o tokenizer de outro modelo para a pasta dele
# (ex.: fazer um fine-tuning novo do 100M sem mexer no atual): "padrão" = a pasta checkpoints de sempre, ou o
# nome de outro PROJETO. Vazio = não copia (um modelo novo, do zero, pela célula 3).
RAMO = "main"  #@param {type:"string"}
NOME = ""  #@param {type:"string"}
PROJETO = ""  #@param {type:"string"}
PARTIR_DE = ""  #@param {type:"string"}
import os, re, shlex, codecs, subprocess, unicodedata, torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "nenhuma (só processador)")

def rodar(comando):
    """Roda um comando mostrando a saída aos poucos; se ele falhar, a célula para com erro."""
    print("$", comando, flush=True)
    p = subprocess.Popen(comando, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         env=dict(os.environ, PYTHONUNBUFFERED="1"))
    texto = codecs.getincrementaldecoder("utf-8")("replace")
    while pedaco := p.stdout.read1(65536):
        print(texto.decode(pedaco), end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"o comando falhou (código {p.returncode}): {comando}")

if not os.path.ismount("/content/drive"):  # pelo app, o Drive já chega montado
    from google.colab import drive
    drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/LLM_Laika"
PROJETO, NOME = PROJETO.strip(), NOME.strip() or PROJETO.strip()
SUFIXO = ""
def sufixo(nome):
    sem_acento = unicodedata.normalize("NFKD", nome).encode("ascii", "ignore").decode().lower()
    return "_" + (re.sub(r"[^a-z0-9]+", "_", sem_acento).strip("_") or "projeto")

if PROJETO:
    SUFIXO = sufixo(PROJETO)
CHECKPOINTS, TOKENS = f"checkpoints{SUFIXO}", f"{DRIVE}/tokens{SUFIXO}"

if not os.path.exists("/content/LLM_Laika"):
    rodar("git clone -q https://github.com/BrunoDevPro/LLM_Laika.git /content/LLM_Laika")
os.chdir("/content/LLM_Laika")
rodar(f"git fetch -q origin {shlex.quote(RAMO)} && git checkout -q -f -B {shlex.quote(RAMO)} FETCH_HEAD")
rodar("pip install -q tokenizers datasets pyarrow fsspec aiohttp")

# As pastas grandes ficam no Drive; aqui são só atalhos
for local, remoto in [("data/books", "books"), ("data/wiki", "wiki"), ("data/qa_auto", "qa_auto"),
                      ("data/carolina", "carolina"), ("data/gigaverbo", "gigaverbo"), ("data/sft", "sft"),
                      ("data/rag", "rag"), ("checkpoints", CHECKPOINTS)]:
    remoto = f"{DRIVE}/{remoto}"
    os.makedirs(remoto, exist_ok=True)
    if os.path.islink(local) and os.readlink(local) != remoto:  # outro PROJETO nesta mesma máquina
        os.remove(local)
    if not os.path.islink(local):
        if os.path.isdir(local) and os.listdir(local):
            rodar(f"cp -rn {local}/. {remoto}/")
        rodar(f"rm -rf {local}")
        os.symlink(remoto, local)

print("Projeto:", PROJETO or "(padrão)", "->", f"{DRIVE}/{CHECKPOINTS}" + (f" · nome no app: {NOME}" if NOME else ""))
if PROJETO:
    pasta = f"{DRIVE}/{CHECKPOINTS}"
    for nome in ["pretrain_best.pt", "tokenizer.json"]:
        # Cópia feita à mão no Drive: "Cópia de pretrain_best.pt", "Copy of tokenizer.json", "pretrain_best (1).pt"
        if not os.path.exists(f"{pasta}/{nome}"):
            base, ext = os.path.splitext(nome)
            for outro in sorted(os.listdir(pasta)):
                if outro in (f"Cópia de {nome}", f"Copy of {nome}") or re.fullmatch(re.escape(base) + r" \(\d+\)" + re.escape(ext), outro):
                    print(f"Renomeando {outro} -> {nome}")
                    os.rename(f"{pasta}/{outro}", f"{pasta}/{nome}")
                    break
    origem = PARTIR_DE.strip()
    if origem and not os.path.exists(f"{pasta}/pretrain_best.pt"):
        de = f"{DRIVE}/checkpoints" + ("" if unicodedata.normalize("NFKD", origem).encode("ascii", "ignore").decode().lower()
                                       in ("padrao", "sempre", "checkpoints") else sufixo(origem))
        for nome in ["pretrain_best.pt", "tokenizer.json"]:
            if not os.path.exists(f"{de}/{nome}"):
                raise RuntimeError(f"PARTIR_DE: não achei {de}/{nome}")
            print(f"Copiando {de}/{nome} -> {pasta}/")
            rodar(f"cp {shlex.quote(de + '/' + nome)} {shlex.quote(pasta + '/')}")
    print("Na pasta do projeto:", sorted(os.listdir(pasta)))
if PROJETO and not os.path.exists(f"{DRIVE}/{CHECKPOINTS}/pretrain_best.pt"):
    print(f"Atenção: o projeto \"{PROJETO}\" ainda não tem pré-treino. Rode a célula 3 (MODO \"do_zero\") antes da 4, "
          "ou use PARTIR_DE = \"padrão\" para partir do pré-treino de sempre. Para só dar um nome ao modelo de sempre, "
          "deixe PROJETO vazio e use NOME.")
print("No Drive:", {p: len(os.listdir(f"{DRIVE}/{p}")) for p in ["books", "wiki", "rag", CHECKPOINTS]})

In [ ]:
# 2) DADOS: livros do Gutenberg + artigos da Wikipédia + índice de busca (RAG)
# Só baixa o que ainda não está no Drive; pedir mais continua de onde parou.
# 1 milhão de artigos: ~3 GB no Drive, de meia hora a uma hora para baixar, e o índice do RAG fica com
# ~4 GB. Não precisa de GPU (pelo app, escolha "Sem GPU" para rodar só esta célula).
# Para o modelo de 100M: CAROLINA (Corpus Carolina, ~2 GB de texto, sem leis e decisões judiciais) e
# GIGAVERBO (textos educativos da web; 1,5 bilhão de tokens ≈ 6 GB no Drive, ~20 min). Os dois só vão
# para o pré-treino, não para o índice do RAG.
# SFT: conversas do GigaVerbo-v2-SFT (Apache 2.0; geradas por outra IA e filtradas por qualidade) para o
# fine-tuning: 60% pedidos variados, 30% perguntas com contexto e 10% resumos. 300 mil ≈ 1 GB no Drive.
# Quem já baixou antes da versão 1.11: escolha de novo o mesmo número, e só os resumos que faltam são baixados.
LIVROS = 500  #@param {type:"integer"}
ARTIGOS = "100 mil"  #@param ["10 mil", "50 mil", "100 mil", "300 mil", "500 mil", "1 milhão", "todos (~1,1 milhão)"]
CAROLINA = False  #@param {type:"boolean"}
GIGAVERBO = "nada"  #@param ["nada", "500 milhões de tokens", "1 bilhão de tokens", "1,5 bilhão de tokens", "2 bilhões de tokens"]
SFT = "nada"  #@param ["nada", "100 mil conversas", "200 mil conversas", "300 mil conversas", "500 mil conversas"]
REFAZER_INDICE = False  #@param {type:"boolean"}
quantos = {"10 mil": 10000, "50 mil": 50000, "100 mil": 100000, "300 mil": 300000, "500 mil": 500000,
           "1 milhão": 1000000, "todos (~1,1 milhão)": -1}[ARTIGOS]
giga = {"nada": 0, "500 milhões de tokens": 5e8, "1 bilhão de tokens": 1e9, "1,5 bilhão de tokens": 1.5e9,
        "2 bilhões de tokens": 2e9}[GIGAVERBO]
sft = {"nada": 0, "100 mil conversas": 100000, "200 mil conversas": 200000, "300 mil conversas": 300000,
       "500 mil conversas": 500000}[SFT]
rodar(f"python prepare_data.py --max-books {LIVROS} --wiki {quantos}" + (" --carolina" if CAROLINA else "")
      + (f" --gigaverbo {giga:.0f}" if giga else "") + (f" --sft {sft}" if sft else ""))
rodar("python rag.py build" + (" --force" if REFAZER_INDICE else ""))

In [ ]:
# 3) PRÉ-TREINO (ensina português). Escolha o MODO:
#   "pular"     -> usa o pré-treino que já está no Drive (padrão; se não existir, treina do zero)
#   "continuar" -> retoma um treino interrompido do ponto em que parou (ex.: o GitHub para em 6 h);
#                  se o último terminou, continua o pré-treino do Drive com os dados atuais (mesmo tokenizer)
#   "do_zero"   -> treina tudo de novo (necessário para trocar o PRESET e para o tokenizer aprender palavras novas)
# PRESET: "colab" (~7M), "colab_big" (~14M), "colab_50m" (~52M, contexto 512, vocabulário 16 mil; use com
# 1 milhão de artigos e, se der, A100 ou L4: na T4 leva ~5 h) ou "colab_100m" (~104M, contexto 2048 com RoPE,
# vocabulário 24 mil; precisa dos dados da célula 2 com CAROLINA e GIGAVERBO, e da A100: ~4–6 h, então rode
# de novo com MODO "continuar" se parar no limite de 6 h). PASSOS = 0 usa o número de passos do preset.
# COMPILAR: torch.compile (~20% mais rápido na A100 depois de 1–2 min compilando).
# Um modelo novo (outro PRESET) sem apagar o atual: dê um PROJETO na célula 1 antes.
MODO = "pular"  #@param ["pular", "continuar", "do_zero"]
PRESET = "colab"  #@param ["tiny", "colab", "colab_big", "colab_50m", "colab_100m"]
PASSOS = 0  #@param {type:"integer"}
COMPILAR = False  #@param {type:"boolean"}
import os
extra = (f" --max-steps {PASSOS}" if PASSOS > 0 else "") + (" --compile" if COMPILAR else "")
extra += f" --tokens-backup {TOKENS}"  # numa máquina nova, copia os tokens em vez de tokenizar de novo
existe = os.path.exists("checkpoints/pretrain_best.pt") or os.path.exists("checkpoints/pretrain_last.pt")
if MODO == "pular" and os.path.exists("checkpoints/pretrain_best.pt"):
    print("Usando o pré-treino que já está no Drive.")
elif MODO == "continuar" and existe:
    rodar(f"python train.py --preset {PRESET} --resume{extra}")
else:
    rodar(f"python train.py --preset {PRESET}{extra}")

In [ ]:
# 4) FINE-TUNING (ensina a responder). Usa o pré-treino do Drive e grava checkpoints/finetuned.pt.
# Se o índice do RAG existir (célula 2), parte das perguntas é treinada com Contexto.
# PERGUNTAS_WIKIPEDIA: perguntas com Contexto tiradas dos artigos ("Onde X nasceu?", "O que X inventou?"...).
# RAG_FRACAO: parte das perguntas que pedem informação que ganha o Contexto do RAG (0 a 1).
# CONHECIMENTO: os artigos mais importantes da Wikipédia viram perguntas SEM Contexto, cada fato de 4 jeitos,
# para o modelo guardar fatos ("automático": 5 mil artigos até 20M de parâmetros, 20 mil acima). Madeira e
# Areia ficam de fora (para testar se a Laika responde sozinha).
# USAR_SFT: usa as conversas do GigaVerbo-v2-SFT baixadas na célula 2 (SFT), se houver.
# A taxa de aprendizado (o tamanho de cada ajuste dos pesos) é escolhida pelo tamanho do modelo.
EPOCAS = 2  #@param {type:"integer"}
USAR_RAG = True  #@param {type:"boolean"}
RAG_FRACAO = 0.7  #@param {type:"number"}
PERGUNTAS_WIKIPEDIA = 40000  #@param {type:"integer"}
CONHECIMENTO = "automático"  #@param ["automático", "nenhum", "5 mil artigos", "10 mil artigos", "20 mil artigos", "50 mil artigos"]
USAR_SFT = True  #@param {type:"boolean"}
artigos = {"automático": -1, "nenhum": 0, "5 mil artigos": 5000, "10 mil artigos": 10000, "20 mil artigos": 20000,
           "50 mil artigos": 50000}[CONHECIMENTO]
rodar(f"python finetune.py --epochs {EPOCAS} --wiki-leitura {PERGUNTAS_WIKIPEDIA} --rag-share {RAG_FRACAO}"
      f" --conhecimento {artigos}" + ("" if USAR_RAG else " --no-rag") + ("" if USAR_SFT else " --sft-max 0"))

In [ ]:
# 5) CONVERSAR [só no navegador] (digite 'sair' para encerrar)
# Comandos: /fontes (mostra de onde veio o contexto), /rag (liga/desliga a busca), /memoria
# A Laika lembra o que você contar (ex.: "meu nome é Ana", "lembre que amanhã tenho prova").
# Para testar só o pré-treino: sys.argv = ["chat.py", "--base"]
import sys
sys.argv = ["chat.py"]
%run chat.py

In [ ]:
# 6) EXPORTAR PARA O CELULAR: um zip só, em Meu Drive/LLM_Laika/laika_pacote.zip
# No app: botão de versões → Escolher arquivos → laika_pacote.zip. Vira uma versão nova
# (a anterior fica nos backups). Pela tela Colab do app, o zip aparece sozinho em "Modelos prontos".
# Com um NOME (ou PROJETO) na célula 1, o zip leva o nome e o app guarda o modelo direto nos backups com
# ele, sem trocar o modelo em uso. Com um PROJETO, o zip também fica no Drive como laika_pacote_<projeto>.zip.
# INDICE (~640 MB) e TEXTOS (livros e artigos, para o celular montar o índice) deixam o zip bem maior.
MODELO = True  #@param {type:"boolean"}
INDICE = False  #@param {type:"boolean"}
TEXTOS = False  #@param {type:"boolean"}
import os, glob, zipfile, shutil
arquivos = []
if MODELO:
    arquivos += [("checkpoints/" + n, n) for n in ["finetuned.pt", "pretrain_best.pt", "tokenizer.json"]
                 if os.path.exists("checkpoints/" + n)]
if INDICE and os.path.exists("data/rag/laika_rag.db"):
    arquivos.append(("data/rag/laika_rag.db", "laika_rag.db"))
if TEXTOS:
    for p in ["books", "wiki", "modern"]:
        arquivos += [(f, p + "/" + os.path.basename(f)) for f in sorted(glob.glob(f"data/{p}/*.txt"))]
    if os.path.exists("data/books/titulos.tsv"):
        arquivos.append(("data/books/titulos.tsv", "books/titulos.tsv"))
if not arquivos:
    raise RuntimeError("Nada para exportar: rode o fine-tuning (célula 4) ou marque INDICE ou TEXTOS.")
with zipfile.ZipFile("/content/laika_pacote.zip", "w", zipfile.ZIP_DEFLATED, compresslevel=1) as z:
    for f, nome in arquivos:
        z.write(f, nome)
    if NOME:  # por último: o app reconhece o tipo do zip pela primeira entrada
        z.writestr("laika_nome.txt", NOME)
destino = f"{DRIVE}/laika_pacote{SUFIXO}.zip"
shutil.copy("/content/laika_pacote.zip", destino)
print(f"Pronto: {len(arquivos)} arquivos, {os.path.getsize(destino) / 2**20:.0f} MB em {destino}")

In [ ]:
# 7) AVALIAR: perguntas fixas (data/avaliacao/), busca do RAG e perplexidade, sem sorteio
# Mede o checkpoints/finetuned.pt e, com COMPARAR_COM_BACKUP, também o backup mais recente (o fine-tuning
# anterior), lado a lado. O resultado vai para Meu Drive/LLM_Laika/checkpoints/avaliacao/ (ou a pasta do PROJETO).
COMPARAR_COM_BACKUP = True  #@param {type:"boolean"}
USAR_RAG = True  #@param {type:"boolean"}
rodar("python avaliar.py" + (" --comparar backup" if COMPARAR_COM_BACKUP else "") + ("" if USAR_RAG else " --sem-rag"))